In [1]:
print("Everything is working fine!")

Everything is working fine!


In [2]:
# import dependencies

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


d:\GENAIBatch\langchain_new\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load documents and split into chunks
loader = PyPDFLoader("D:\\GENAIBatch\\langchain_new\\RAG\\PEFT2312.12148v1.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

In [4]:
print(len(docs))

154


In [5]:
# Embeddings model
embeddings = OllamaEmbeddings(
    model = "nomic-embed-text-v2-moe",
    dimensions=768
)

In [6]:
embeddings

OllamaEmbeddings(model='nomic-embed-text-v2-moe', dimensions=768, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [7]:
vdb = Chroma(
    persist_directory="./chroma_db2",
    embedding_function=embeddings
)

C:\Users\anant\AppData\Local\Temp\ipykernel_19356\3242060151.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb = Chroma(


In [8]:
vectorstore = vdb.from_documents(
    documents=docs, 
    embedding=embeddings,
    persist_directory="./chroma_db2",
    collection_name="ollama_test")

In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    seach_kwargs={"k": 3}
)

In [10]:
retriever.invoke("What is the main topic of the paper?")

[Document(metadata={'author': '', 'source': 'D:\\GENAIBatch\\langchain_new\\RAG\\PEFT2312.12148v1.pdf', 'moddate': '2023-12-20T02:07:04+00:00', 'creationdate': '2023-12-20T02:07:04+00:00', 'trapped': '/False', 'creator': 'LaTeX with hyperref', 'page': 19, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': '', 'page_label': '20', 'total_pages': 20, 'subject': '', 'keywords': '', 'producer': 'pdfTeX-1.40.25'}, page_content='Amer. Chapter Assoc. Comput. Linguistics: Hum. Lang. Technol., 2022,\npp. 1791–1799.\n[110] M. T ¨anzer, S. Ruder, and M. Rei, “Memorisation versus generalisa-\ntion in pre-trained language models,” in Proc. Annu. Meeting Assoc.\nComput. Linguistics, 2022, pp. 7564–7578.\n[111] N. Gu, P. Fu, X. Liu, Z. Liu, Z. Lin, and W. Wang, “A gradient control\nmethod for backdoor attacks on parameter-efficient tuning,” in Proc.\nAnnu. Meeting Assoc. Comput. Linguistics , 2023, pp. 3508–3520.\n[112] B. Zhu, Y . Qin

In [11]:
local_llm =ChatOllama(
    model = 'gemma2:9b',
    max_tokens=512,
    temperature=0
)

In [12]:
prompt_template = """Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know."""


In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(prompt_template)

prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know.")

In [14]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=local_llm,
    chain_type = "stuff",
    retriever=retriever)

In [15]:
qa_chain.invoke("What is the main topic of the paper?")

{'query': 'What is the main topic of the paper?',
 'result': "The provided text snippets are about various techniques related to natural language processing (NLP), including:\n\n* **Memorization vs. generalization in pre-trained language models**\n* **Backdoor attacks on parameter-efficient fine-tuning**\n* **Moderate-fitting as a backdoor defender**\n* **Trojan attacks on parameter-efficient fine-tuning**\n* **The expressive power of low-rank adaptation**\n* **Neural network pruning**\n* **Adaptive sparsity by fine-tuning**\n* **Black-box optimization**\n* **Editing models with task arithmetic**\n* **Composing parameter-efficient fine-tuning**\n\n\n\nIt's difficult to pinpoint a single main topic across all these snippets. They collectively explore different aspects of NLP, focusing on model training, security vulnerabilities, and optimization techniques. \n"}